# MapReduce Word Count - Discussion Notebook

**Name:** Jack Ashraf  
**Student ID:** 231006166

A small word-count MapReduce engine using only Python's standard library.

- `NUM_MAPPERS` mapper threads divide the input files between them.
- `NUM_REDUCERS` reducers each write their own `reducer_N.txt`.
- All reducer outputs are merged into one sorted `output/final_output.txt`.

**Pipeline:** `Split -> Map -> Combine -> Partition -> Shuffle & Sort -> Reduce -> Merge`

> Change `NUM_MAPPERS` and `NUM_REDUCERS` in the first code cell, then run all cells.


## 1. Map and combine

Each mapper receives a group of input files, emits `(word, 1)` for every word, and combines repeated words locally.

In [1]:
"""A small word-count MapReduce engine using only Python's standard library."""

import re  # for regular expressions
from pathlib import Path  # for working with file paths
from threading import Thread


NUM_MAPPERS = 5   # number of mapper threads (input groups)
NUM_REDUCERS = 2  # number of reducer output files


def map_words(files):
    """Map: read this mapper's files and emit (word, 1) for each occurrence."""
    for split in files:
        with split.open(encoding="utf-8") as source:
            for line in source:
                for word in re.findall(r"[a-z]+", line.lower()):
                    yield word, 1


def combine(pairs):
    """Combine: count repeated words inside this mapper only."""
    subtotals = {}
    for word, value in pairs:
        subtotals[word] = subtotals.get(word, 0) + value
    return subtotals

## 2. Partition and mapper threads

The stable hash sends every word to exactly one reducer. `run_mapper` is the function each mapper thread runs.

In [2]:
def reducer_for(word):
    # Stable across runs: Python's built-in hash() can change between runs.
    return sum(ord(letter) for letter in word) % NUM_REDUCERS


def partition(subtotals):
    """Partition: assign each local subtotal to exactly one reducer."""
    buckets = [[] for _ in range(NUM_REDUCERS)]
    for word, count in subtotals.items():
        buckets[reducer_for(word)].append((word, count))
    return buckets


def run_mapper(files, index, results, errors):
    try:
        results[index] = partition(combine(map_words(files)))
    except Exception as error:
        errors[index] = error

## 3. Shuffle, reduce, and merge

The shuffle gathers every mapper's pairs for each reducer and groups the counts. Each reducer writes a sorted file. Finally, all reducer files are merged into `final_output.txt`.

In [3]:
def shuffle_and_sort(mapper_results):
    """Shuffle: gather matching partitions; sort and group by word."""
    shuffled = []
    for reducer_id in range(NUM_REDUCERS):
        pairs = []
        for buckets in mapper_results:
            pairs.extend(buckets[reducer_id])
        pairs.sort(key=lambda pair: pair[0])

        groups = {}
        for word, count in pairs:
            if word not in groups:
                groups[word] = []
            groups[word].append(count)
        shuffled.append(groups)
    return shuffled


def reduce_words(groups, output_file):
    """Reduce: sum each word's subtotals and write sorted output."""
    with output_file.open("w", encoding="utf-8") as output:
        for word, values in groups.items():
            total = 0
            for value in values:
                total += value
            output.write(f"{word}\t{total}\n")


def merge_reducers(output_dir):
    """Merge every reducer output file into one sorted final_output.txt."""
    totals = {}
    for reducer_id in range(NUM_REDUCERS):
        reducer_file = output_dir / f"reducer_{reducer_id}.txt"
        with reducer_file.open(encoding="utf-8") as source:
            for line in source:
                word, count = line.rstrip("\n").split("\t")
                totals[word] = totals.get(word, 0) + int(count)

    final_file = output_dir / "final_output.txt"
    with final_file.open("w", encoding="utf-8") as output:
        for word in sorted(totals):
            output.write(f"{word}\t{totals[word]}\n")
    return final_file

## 4. Run and main

The input files are divided round-robin between the mapper threads. If `NUM_MAPPERS` is larger than the number of input files, it is reduced to the file count.

Try `NUM_MAPPERS = 3` to see the five files grouped across three mapper threads.

In [4]:
def run(input_dir, output_dir):
    input_dir, output_dir = Path(input_dir), Path(output_dir)
    if not input_dir.is_dir():
        raise ValueError(f"Input folder does not exist: {input_dir}")
    if input_dir.resolve() == output_dir.resolve():
        raise ValueError("Input and output folders must be different.")
    if NUM_MAPPERS < 1 or NUM_REDUCERS < 1:
        raise ValueError("NUM_MAPPERS and NUM_REDUCERS must be at least 1.")

    files = sorted(path for path in input_dir.glob("*.txt") if path.is_file())
    if not files:
        raise ValueError(f"No .txt files found in {input_dir}")

    # Divide the input files across the mapper threads.
    num_mappers = min(NUM_MAPPERS, len(files))
    splits = [files[index::num_mappers] for index in range(num_mappers)]

    results = [None] * num_mappers
    errors = [None] * num_mappers
    threads = [
        Thread(target=run_mapper, args=(splits[index], index, results, errors))
        for index in range(num_mappers)
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()  # All mapper results must be ready before shuffle.
    for error in errors:
        if error is not None:
            raise error

    shuffled = shuffle_and_sort(results)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Remove reducer files left from an earlier run with a different count.
    for old_file in output_dir.glob("reducer_*.txt"):
        old_file.unlink()
    for reducer_id, groups in enumerate(shuffled):
        output_file = output_dir / f"reducer_{reducer_id}.txt"
        reduce_words(groups, output_file)
        print(f"Reducer {reducer_id}: {len(groups)} unique words -> {output_file}")

    final_file = merge_reducers(output_dir)
    print(f"Merged output: {final_file}")
    print(f"Finished: {num_mappers} mapper threads, {NUM_REDUCERS} reducers.")


def main():
    run("input", "output")


if __name__ == "__main__":
    main()

Reducer 0: 91 unique words -> output\reducer_0.txt
Reducer 1: 91 unique words -> output\reducer_1.txt
Merged output: output\final_output.txt
Finished: 5 mapper threads, 2 reducers.


## 5. Final output

The merged file contains one `word<TAB>count` line per word, sorted alphabetically.

In [5]:
final_file = Path("output/final_output.txt")
lines = final_file.read_text(encoding="utf-8").splitlines()
print(f"{len(lines)} unique words")
print("First 10 lines:")
for line in lines[:10]:
    print(line)

182 unique words
First 10 lines:
a	1
accuracy	1
across	1
advanced	1
aggregate	1
aggregation	1
all	1
an	1
analysis	2
analytics	2


## Discussion points

1. **Mappers:** `NUM_MAPPERS` controls how many mapper threads run. The input files are split between them.
2. **Map and combine:** each mapper emits `(word, 1)` and combines repeated words locally.
3. **Partition:** the stable character-code hash sends each word to exactly one reducer.
4. **Shuffle and reduce:** reducers sum the grouped counts and write `reducer_0.txt`, `reducer_1.txt`, and so on.
5. **Merge:** the reducer files are read and merged into one sorted `final_output.txt`.
6. **Configurable:** changing `NUM_MAPPERS` or `NUM_REDUCERS` changes the number of threads and output files, but the final word counts stay the same.
